# 🤖 Room Booking Prediction — ML Module

## College Resource Booking System

This notebook trains **3 ML models** to predict room booking patterns:

1. **Random Forest Classifier** — Ensemble learning
2. **Decision Tree Classifier** — Interpretable classification
3. **Logistic Regression** — Probability estimation

### Dataset Features
| Feature | Description |
|---------|-------------|
| user | Student/Faculty ID |
| room | Room number (1001-4410) |
| slot | Time slot (9-10, 10-11, 11-12, 12-1, 2-3, 3-4) |
| booked | Target variable (0=Not booked, 1=Booked) |

In [ ]:
# Import Libraries
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 📊 Step 1: Generate Synthetic Dataset (1500+ rows)

In [ ]:
# Configuration
VALID_SLOTS = ['9-10', '10-11', '11-12', '12-1', '2-3', '3-4']
N_SAMPLES = 1500
random.seed(42)
np.random.seed(42)

# Generate rooms 1001–4410
all_rooms = []
for floor in range(1, 5):
    for room in range(1, 411):
        all_rooms.append(floor * 1000 + room)

print(f'Total valid rooms: {len(all_rooms)}')
print(f'Room range: {min(all_rooms)} → {max(all_rooms)}')

# Generate users
students = [f'S{i}' for i in range(1001, 1051)]
faculty = [f'F{i}' for i in range(101, 121)]
all_users = students + faculty
print(f'Total users: {len(all_users)} ({len(students)} students, {len(faculty)} faculty)')

In [ ]:
# Generate Dataset
popular_rooms = random.sample(all_rooms, 300)

data = []
for _ in range(N_SAMPLES):
    user = random.choice(all_users)
    room = random.choice(popular_rooms)
    slot_idx = random.randint(0, len(VALID_SLOTS) - 1)
    slot = VALID_SLOTS[slot_idx]
    
    floor_num = room // 1000
    is_student = 1 if user.startswith('S') else 0
    
    # Booking probability influenced by features
    prob = 0.5
    if slot_idx < 3: prob += 0.15          # Morning slots more popular
    prob += (5 - floor_num) * 0.03         # Lower floors slightly more popular
    if is_student: prob += 0.05            # Students book more
    prob += random.uniform(-0.2, 0.2)      # Random noise
    prob = max(0, min(1, prob))
    
    booked = 1 if random.random() < prob else 0
    
    data.append({
        'user': user,
        'room': room,
        'slot': slot,
        'slot_idx': slot_idx,
        'floor': floor_num,
        'is_student': is_student,
        'booked': booked
    })

df = pd.DataFrame(data)
print(f'\nDataset shape: {df.shape}')
print(f'\nFirst 10 rows:')
df.head(10)

In [ ]:
# Dataset Statistics
print('=== Dataset Summary ===')
print(f'Total samples: {len(df)}')
print(f'\nBooking distribution:')
print(df['booked'].value_counts())
print(f'\nBooking rate: {df["booked"].mean():.2%}')
print(f'\nSlot distribution:')
print(df['slot'].value_counts())
print(f'\nFloor distribution:')
print(df['floor'].value_counts().sort_index())

## 🔧 Step 2: Data Preprocessing

In [ ]:
# Feature Selection
feature_cols = ['room', 'slot_idx', 'floor', 'is_student']
X = df[feature_cols].values
y = df['booked'].values

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')
print(f'\nFeatures: {feature_cols}')
print(f'Target: booked (binary)')

## 🌲 Step 3: Train Model 1 — Random Forest

In [ ]:
# Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
rf_cm = confusion_matrix(y_test, rf_pred)

print('=== Random Forest Classifier ===')
print(f'Accuracy: {rf_acc:.4f}')
print(f'\nConfusion Matrix:')
print(rf_cm)
print(f'\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=['Not Booked', 'Booked']))

# Feature Importance
print('Feature Importance:')
for feat, imp in zip(feature_cols, rf.feature_importances_):
    print(f'  {feat}: {imp:.4f}')

## 🌳 Step 4: Train Model 2 — Decision Tree

In [ ]:
# Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42, max_depth=8)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
dt_cm = confusion_matrix(y_test, dt_pred)

print('=== Decision Tree Classifier ===')
print(f'Accuracy: {dt_acc:.4f}')
print(f'\nConfusion Matrix:')
print(dt_cm)
print(f'\nClassification Report:')
print(classification_report(y_test, dt_pred, target_names=['Not Booked', 'Booked']))

## 📈 Step 5: Train Model 3 — Logistic Regression

In [ ]:
# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
lr_cm = confusion_matrix(y_test, lr_pred)

print('=== Logistic Regression ===')
print(f'Accuracy: {lr_acc:.4f}')
print(f'\nConfusion Matrix:')
print(lr_cm)
print(f'\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['Not Booked', 'Booked']))

## 📊 Step 6: Model Comparison & Visualization

In [ ]:
# Accuracy Comparison
models_data = {
    'Model': ['Random Forest', 'Decision Tree', 'Logistic Regression'],
    'Accuracy': [rf_acc, dt_acc, lr_acc]
}
comparison_df = pd.DataFrame(models_data)

print('\n' + '='*50)
print('  MODEL ACCURACY COMPARISON')
print('='*50)
print(comparison_df.to_string(index=False))

# Best model
best_idx = comparison_df['Accuracy'].idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f'\n★ Best Model: {best_model_name} (Accuracy: {best_accuracy:.4f})')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3b82f6', '#10b981', '#f59e0b']
axes[0].barh(comparison_df['Model'], comparison_df['Accuracy'], color=colors)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xlim(0, 1)
for i, v in enumerate(comparison_df['Accuracy']):
    axes[0].text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')

# Confusion matrices
all_cms = [rf_cm, dt_cm, lr_cm]
all_names = ['Random Forest', 'Decision Tree', 'Logistic Regression']

# Show best model's confusion matrix
best_cm = all_cms[best_idx]
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Not Booked', 'Booked'],
            yticklabels=['Not Booked', 'Booked'])
axes[1].set_title(f'Confusion Matrix — {best_model_name}')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nChart saved as model_comparison.png')

In [ ]:
# All confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (cm, name) in enumerate(zip(all_cms, all_names)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', ax=axes[i],
                xticklabels=['Not Booked', 'Booked'],
                yticklabels=['Not Booked', 'Booked'])
    acc = [rf_acc, dt_acc, lr_acc][i]
    axes[i].set_title(f'{name}\nAccuracy: {acc:.4f}')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved as confusion_matrices.png')

## 💾 Step 7: Save Best Model

In [ ]:
# Select best model
best_models = {'Random Forest': rf, 'Decision Tree': dt, 'Logistic Regression': lr}
best_model = best_models[best_model_name]

# Save model
model_path = 'best_model.pkl'
joblib.dump(best_model, model_path)

print(f'\n✓ Best model ({best_model_name}) saved to: {model_path}')
print(f'  Accuracy: {best_accuracy:.4f}')
print(f'  File size: {os.path.getsize(model_path) / 1024:.1f} KB')

# Verify by loading
loaded_model = joblib.load(model_path)
verify_pred = loaded_model.predict(X_test)
verify_acc = accuracy_score(y_test, verify_pred)
print(f'  Verification accuracy: {verify_acc:.4f} ✓')

In [ ]:
# Test prediction
print('\n=== Sample Predictions ===')
test_cases = [
    [1001, 0, 1, 1],  # Room 1001, slot 9-10, floor 1, student
    [2205, 3, 2, 0],  # Room 2205, slot 12-1, floor 2, faculty
    [4100, 5, 4, 1],  # Room 4100, slot 3-4, floor 4, student
]

for case in test_cases:
    pred = loaded_model.predict([case])[0]
    prob = loaded_model.predict_proba([case])[0]
    slot_name = VALID_SLOTS[case[1]]
    print(f'  Room {case[0]}, Slot {slot_name}: '
          f'{"Likely Booked" if pred == 1 else "Likely Available"} '
          f'(prob: {max(prob):.3f})')

## 📋 Summary

### Results
- **3 models trained**: Random Forest, Decision Tree, Logistic Regression
- **Best model saved** as `best_model.pkl`
- **Confusion matrices** generated for all models
- **Flask integration** ready via `predict()` endpoint

### Architecture
```
Web UI (HTML/CSS/JS)
        ↓
Flask Backend (Python)
        ↓
C++ OOP Engine
        ↓
SQLite Database
        ↓
ML Models (This Notebook)
```

In [ ]:
# Save training data CSV
df.to_csv('training_data.csv', index=False)
print(f'Training data saved: training_data.csv ({len(df)} rows)')
print('\n✅ ML Module Complete!')